# Drainage divide networks and competing basin pairs with pydivide (Python)

## Authors

- Daniel Peifer, Graduate Program in Geography, Institute of Geosciences, Federal University of Minas Gerais, 31270-901 Belo Horizonte, Brazil ([peiferdaniel@gmail.com](mailto:peiferdaniel@gmail.com), [GitHub](https://github.com/peiferdaniel))

This notebook is licensed under [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Highlighted references

Scherler, D. and Schwanghart, W.: Drainage divide networks – Part 1: Identification and ordering in digital elevation models, Earth Surf. Dynam., 8, 245–259, https://doi.org/10.5194/esurf-8-245-2020, 2020.

Scherler, D. and Schwanghart, W.: Drainage divide networks – Part 2: Response to perturbations, Earth Surf. Dynam., 8, 261–274, https://doi.org/10.5194/esurf-8-261-2020, 2020.

## Audience

Users of TopoToolbox who work on drainage divides, divide mobility and river capture. Readers should be comfortable running a Python notebook and familiar with DEMs, stream networks and drainage-area thresholds.

## Introduction

Drainage divides separate neighbouring river basins, but comparing those basins requires choices about channel thresholds, analysis boundaries and the reference level for χ. Scherler and Schwanghart's `DIVIDEobj` introduced divide-network analysis to TopoToolbox (Scherler and Schwanghart, 2020a, b), and Scherler's 2020 tutorial on the Big Tujunga DEM, [Introduction to @DIVIDEobj](https://topotoolbox.wordpress.com/2020/05/29/introduction-to-divideobj/), is the starting point here. [pydivide](https://github.com/ATLAS-Topo/pydivide) brings that workflow to Python, on pytopotoolbox, and extends it with signed basin contrasts, recorded comparison settings, basin-pair analysis, and profiles along and across a shared divide. It was written for large DEMs (cell and node indices use 64-bit integers), but this example needs none of that.

We first use the original Big Tujunga DEM to introduce the setting. The main analysis uses a larger Copernicus GLO-30 DEM while retaining the familiar map window. This provides downstream context for comparisons between neighbouring drainage systems. We ask:

1. How do we construct, order and inspect a divide network?
2. Which basin contrasts meet the chosen χ-reference conditions, and how do we read their signs?
3. How can a basin pair and its profiles help us inspect those contrasts in the landscape?

The example takes a few minutes once its dependencies and data are available, depending on the computer used.

### Dependencies

Besides pytopotoolbox and matplotlib, this example needs pydivide, which installs from GitHub together with geopandas, rasterio and shapely, and cmocean for some of the colour maps:

```bash
pip install "pydivide[plot] @ git+https://github.com/ATLAS-Topo/pydivide@v0.2.0" cmocean
```

It downloads two tiles of the Copernicus GLO-30 DEM, about 80 MB, from their public bucket on Amazon Web Services. No account or key is needed.

In [ ]:
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path

import cmocean
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, LogNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoLocator, FuncFormatter, MaxNLocator
from rasterio.crs import CRS
from rasterio.merge import merge

import topotoolbox as tt
import pydivide
from pydivide import divideops as ops
from pydivide.divmetrics import edge_cells
from pydivide.network import fx_grid
from pydivide.plotting import plot_basin_pair

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.labelsize": 9, "legend.fontsize": 8, "legend.frameon": True,
    "legend.framealpha": 0.88, "legend.edgecolor": "none",
})
RIVER = dict(color="#0a5cff", linewidth=0.9,
             path_effects=[pe.Stroke(linewidth=2.2, foreground="w", alpha=0.75), pe.Normal()])
out = Path(tempfile.mkdtemp())
print("pydivide", pydivide.__version__, "| exports are written to", out)

## The Big Tujunga DEM, and a larger one around it

Big Tujunga Canyon, in the San Gabriel Mountains of California, is the example DEM of TopoToolbox and of the 2020 tutorial. We load it with `load_dem`, route the flow with D8 and carving, and define channels by a drainage area of 1,000 cells (0.9 km² at 30 m), as the tutorial does with `minarea`. The extent of this DEM is the window of every map that follows. Every river system of a DEM ends at an outlet, and the first map shows where they end.

In [ ]:
bt = tt.load_dem("bigtujunga")
W = bt.extent                                   # the window of the tutorial, kept for every map
km = FuncFormatter(lambda v, _: f"{v / 1000:.0f}")
THRESHOLD = dict(threshold=1000, units="pixels")        # the channel threshold of the tutorial
fd0 = tt.FlowObject(bt)
st0 = tt.StreamObject(fd0, **THRESHOLD)


def outlets(st, dem):
    # cells and elevations of the outlets of every river system
    cells = np.asarray(st.stream)[np.asarray(st.streampoi("outlets"))]
    return cells, np.asarray(dem.z).ravel()[cells]


oc0, oz0 = outlets(st0, bt)
print(f"{len(oc0)} river systems leave this DEM between {oz0.min():.0f} and {oz0.max():.0f} m. "
      f"{(oz0 <= 400).sum()} of them leave it at or below 400 m.")

In [ ]:
r0, c0 = np.divmod(oc0, bt.shape[1])
ox0, oy0 = W[0] + (c0 + 0.5) * bt.cellsize, W[3] - (r0 + 0.5) * bt.cellsize

fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
bt.plot_hs(ax=ax, cmap=cmocean.tools.crop_by_percent(cmocean.cm.topo, 50, which="min"), exaggerate=1.5)
st0.plot(ax=ax, **RIVER)
sc = ax.scatter(ox0, oy0, c=oz0, s=90, cmap=cmocean.cm.thermal, vmin=300, vmax=1500, edgecolor="k", zorder=6)
for x, y, z in zip(ox0, oy0, oz0):
    if z <= 400:
        ax.annotate(f"{z:.0f} m", (x, y), xytext=(12, 4), textcoords="offset points", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=3, foreground="w")], zorder=7)
ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
ax.set_xlabel("Easting (km)"); ax.set_ylabel("Northing (km)")
fig.colorbar(sc, ax=ax, shrink=0.8, label="Elevation of the outlet (m)")
ax.set_title("Big Tujunga: the channels of the tutorial and where each river system leaves the DEM");

One choice has to be made before the divides are compared. χ, the coordinate that linearises river profiles (Perron and Royden, 2013), is integrated upstream from a reference elevation, the base level. Two basins can be compared through χ only when both are referenced to the same one. pydivide takes the base level as an explicit argument of its metrics call and flags every divide whose two sides do not reach it. The DEM that comes with TopoToolbox is a cut around the Big Tujunga catchment. Its own river leaves the DEM at 315 m, but most of the other river systems in the cut are truncated by the edge of the DEM hundreds of metres higher, as the map shows. At a reference of 400 m there is almost nothing left to compare the main basin with.

We therefore download the two Copernicus GLO-30 tiles that cover the area, cut the box 34.10–34.60° N, 118.60–117.80° W (about 70 × 55 km) out of them, and project it to the UTM zone of the Big Tujunga DEM at 30 m. The neighbouring rivers then reach a common reference of 400 m inside the DEM. Everything is computed on the larger DEM and shown in the original window. The box is widened to whole pixels of the tiles, which are centred on whole arc seconds, because a box that splits the edge pixels leaves the last column empty in some versions of rasterio.

In [ ]:
south, north, west, east = 34.10, 34.60, -118.60, -117.80
url = "https://copernicus-dem-30m.s3.amazonaws.com/{0}/{0}.tif"
names = [f"Copernicus_DSM_COG_10_N34_00_W{w}_00_DEM" for w in ("119", "118")]
local = []
for n in names:                                         # plain HTTPS downloads, retried if the connection drops
    path = out / f"{n}.tif"
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(url.format(n), path)
            break
        except (urllib.error.URLError, OSError) as err:     # a dropped connection is an OSError
            if attempt == 2:
                raise
            print(f"download of {n} failed ({err}), retrying")
            time.sleep(5)
    local.append(path)
    print(f"{n}: {path.stat().st_size / 1e6:.1f} MB")

tiles = [rasterio.open(p) for p in local]
t = tiles[0].transform                                  # widen the box to whole source pixels
left, right = (t.c + t.a * f((v - t.c) / t.a) for v, f in ((west, np.floor), (east, np.ceil)))
top, bottom = (t.f + t.e * f((v - t.f) / t.e) for v, f in ((north, np.floor), (south, np.ceil)))
z, transform = merge(tiles, bounds=(left, bottom, right, top))
profile = tiles[0].profile | dict(driver="GTiff", height=z.shape[1], width=z.shape[2], transform=transform)
with rasterio.open(out / "bigtujunga_large.tif", "w", **profile) as dst:
    dst.write(z)

dem = tt.read_tif(str(out / "bigtujunga_large.tif")).reproject(CRS.from_epsg(32611), resolution=30.0)
fd = tt.FlowObject(dem)
st = tt.StreamObject(fd, **THRESHOLD)                    # channels from 1,000 cells, as in the tutorial
dem

A few small helpers keep the maps consistent: a grey hillshade with the channels underneath, axes in kilometres cropped to the window, and a way to draw a set of divide lines coloured by a value. The channels drawn on every map are those of the 1,000-cell network above, held fixed for context whatever the map shows. The raster extent is taken from the transform of the DEM. After `reproject`, the bounds that `extent` reports fall short of the grid by up to one cell, and a raster drawn with them would be squeezed against the divide nodes, which follow the transform.

In [ ]:
HS = dem.hillshade()
T = dem.transform
EXT = (T.c, T.c + T.a * dem.shape[1], T.f + T.e * dem.shape[0], T.f)   # left, right, bottom, top


def km_axes(ax, extent=W):
    # axes in kilometres, cropped to a window
    ax.xaxis.set_major_locator(AutoLocator()); ax.yaxis.set_major_locator(AutoLocator())
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.set_xlabel("Easting (km)"); ax.set_ylabel("Northing (km)")
    ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:]); ax.set_aspect("equal")


def basemap(ax, alpha=0.55, extent=W, rivers=True):
    # hillshade, the channels for context, then the window
    HS.plot(ax=ax, extent=EXT, cmap="Greys_r", vmin=0, vmax=1, alpha=alpha)
    if rivers:
        st.plot(ax=ax, color="#4d88ff", linewidth=0.45, alpha=0.9, zorder=2)
    km_axes(ax, extent)
    return ax


def divide_lines(ax, gdf, values=None, cmap="viridis", norm=None, widths=0.8, color="k", zorder=4, **kw):
    lc = LineCollection([np.asarray(g.coords) for g in gdf.geometry],
                        linewidths=widths, zorder=zorder, capstyle="round", joinstyle="round", **kw)
    if values is None:
        lc.set_color(color)
    else:
        lc.set_array(np.asarray(values, dtype=float)); lc.set_cmap(cmap); lc.set_norm(norm)
    ax.add_collection(lc)
    return lc


def window_rectangle(ax):
    ax.add_patch(mpl.patches.Rectangle((W[0], W[2]), W[1] - W[0], W[3] - W[2], fill=False, ec="k", lw=1.6, zorder=8))

Which river systems drain the window, and where do they leave the larger DEM? `drainagebasins` delineates the basin of every outlet.

In [ ]:
oc, oz = outlets(st, dem)
db = np.asarray(fd.drainagebasins(outlets=oc).z)
z_of_basin = dict(zip(db.ravel()[oc].tolist(), oz.tolist()))
xs = EXT[0] + (np.arange(dem.shape[1]) + 0.5) * dem.cellsize
ys = EXT[3] - (np.arange(dem.shape[0]) + 0.5) * dem.cellsize
in_window = np.outer((ys >= W[2]) & (ys <= W[3]), (xs >= W[0]) & (xs <= W[1]))
systems = [b for b in np.unique(db[in_window]) if b > 0 and (db[in_window] == b).sum() > 1000]
for b in sorted(systems, key=z_of_basin.get):
    print(f"river system {b:3d}: {(db[in_window] == b).sum() * 9e-4:7.1f} km² of the window, leaves the DEM at {z_of_basin[b]:5.0f} m")
z_north = max(z_of_basin[b] for b in systems)           # the northern system leaves the DEM highest

In [ ]:
r, cc = np.divmod(oc, dem.shape[1])
ox, oy = EXT[0] + (cc + 0.5) * dem.cellsize, EXT[3] - (r + 0.5) * dem.cellsize
lab_idx = np.full(db.shape, np.nan)
for i_, b_ in enumerate(sorted(systems, key=z_of_basin.get)):
    lab_idx[db == b_] = i_
pal = ListedColormap(["#a6cee3", "#b2df8a", "#fdbf6f", "#cab2d6", "#fb9a99", "#ffff99", "#e31a1c"][:len(systems)])

fig, ax = plt.subplots(figsize=(10, 7.6), layout="constrained")
basemap(ax, alpha=0.7, extent=EXT, rivers=False)
ax.imshow(lab_idx, extent=EXT, cmap=pal, alpha=0.55, interpolation="nearest", vmin=-0.5, vmax=len(systems) - 0.5)
st.plot(ax=ax, **RIVER)
window_rectangle(ax)
first = {b: np.flatnonzero(db.ravel()[oc] == b)[0] for b in systems}      # the outlet of each system
for n, b in enumerate(sorted(systems, key=lambda b: ox[first[b]])):
    k = first[b]
    ax.scatter(ox[k], oy[k], s=110, color="k", zorder=7, edgecolor="w", linewidth=1.5)
    if oy[k] < EXT[2] + 500:                 # on the southern edge: above the point, staggered
        off, ha = (0, 12 + 14 * (n % 2)), "center"
    elif oy[k] > EXT[3] - 500:               # on the northern edge: below the point
        off, ha = (0, -18), "center"
    else:                                    # on a side: to the right
        off, ha = (10, 0), "left"
    ax.annotate(f"{oz[k]:.0f} m", (ox[k], oy[k]), xytext=off, textcoords="offset points", fontweight="bold",
                ha=ha, va="center", path_effects=[pe.withStroke(linewidth=3, foreground="w")], zorder=8)
km_axes(ax, EXT)
ax.set_title("The larger DEM: the river systems that drain the window (box) and where they leave the DEM", pad=12);

All but one of these systems drain to the Pacific and leave the larger DEM below 400 m. The one in the north (yellow) drains northward, towards the Mojave Desert side of the range, and its outlet leaves this DEM at about 788 m, above the 400 m reference. We will see pydivide flag the divides against it.

## The divide network in one call

The map the tutorial builds up to is the divide network with its edges cleaned, sorted and ordered, drawn where the divide distance exceeds 1 km with the line width by topological order: `DIVIDEobj`, `cleanedges`, `divorder` and `plot(D, 'limit', [1000 inf])`. In pydivide the network, its sorting, the divide distance and the three divide orders come from one call, `divide_network`, and the tutorial's map is a filter on the pieces it exports. The network follows the boundaries between reach basins along cell corners and is split into segments at junctions. The divides along the edge of the larger DEM lie far outside the window, so nothing needs cleaning for this view. Every map that follows keeps the same view, divide distance of 1 km and more, so that the products can be compared at a glance.

In [ ]:
D = pydivide.divide_network(fd=fd, st=st, verbose=False)      # the whole network, sorted, with distance and orders
print(f"{D.n_segments:,} segments, {len(D.edge_nodes):,} divide edges, "
      f"{len(D.endpoints):,} endpoints, {len(D.junctions):,} junctions")

LIMIT = 1000.0                                                # plot(D, 'limit', [1000 inf]): the view of every map below
tree = D.to_geodataframe(seglength=60, scheme="topo")
tree = tree[tree.distmin >= LIMIT]
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
st.plot(ax=ax, color="#4d88ff", linewidth=0.45, alpha=0.9, zorder=2)
divide_lines(ax, tree, color="k", widths=0.3 + 4.2 * (tree["order"] / tree["order"].max()) ** 0.8)
km_axes(ax)
ax.set_title("divide_network, one call: the divide network of the tutorial, distance ≥ 1 km, width by topological order");

The rest of the notebook looks at what this call produced, and then measures across the divides.

## Edge effects: the `isbnd` flag and `clean_edges`

Divides that run along the edge of a DEM belong to basins the DEM cuts off. The tutorial removes them with `cleanedges`. pydivide offers both. Every exported piece carries a flag `isbnd` that is 1 when it touches the edge of the DEM or nodata, so you can decide later what to drop, and `pydivide.divideops.clean_edges` reproduces `cleanedges`, like the other methods of `DIVIDEobj` (more on them at the end). The filters need the grid of diagonal flow crossings, which is built from the flow edges. With the larger DEM, the flagged pieces lie far from the window.

In [ ]:
pieces = D.to_geodataframe(seglength=300)
FX = fx_grid(D.grid, np.asarray(fd.source, np.int64), np.asarray(fd.target, np.int64))
Dc = ops.clean_edges(D, FX)
print(f"{len(pieces):,} pieces of about 300 m, {int(pieces.isbnd.sum()):,} of them flagged isbnd. "
      f"After clean_edges the network is re-segmented into {Dc.n_segments:,} segments.")

fig, ax = plt.subplots(figsize=(10, 7.6), layout="constrained")
basemap(ax, extent=EXT)
divide_lines(ax, pieces[(pieces.isbnd == 0) & (pieces.distmin >= LIMIT)], color="#3b3b3b", widths=0.5)
divide_lines(ax, pieces[pieces.isbnd == 1], color="#d7301f", widths=1.8, zorder=5)
window_rectangle(ax)
ax.legend(handles=[Line2D([], [], color="#3b3b3b", lw=1.2, label="interior pieces, distance ≥ 1 km"),
                   Line2D([], [], color="#d7301f", lw=2, label="isbnd = 1: truncated by the edge of the DEM"),
                   Line2D([], [], color="k", lw=1.6, label="the window of the tutorial")], loc="lower left")
ax.set_title("Pieces of divide flagged as truncated by the edge of the larger DEM");

## Divide distance and the three divide orders

Divide distance is the length along the network from the endpoints, inherited down the divide tree. Divide order can be Strahler, Shreve or topological. As in `DIVIDEobj`, both are values of the node instances, aligned with `D.network.seg_nodes`, and the exports carry them per feature, with the minimum and maximum distance inside each piece (`distmin`, `distmax`). That is what made the opening map a filter on the pieces (`distmin >= 1000`). Here is the same tree coloured by divide distance, and the three orders on the cleaned network.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
basemap(ax, alpha=0.45)
w = 0.4 + 4.0 * (tree["order"] / tree["order"].max()) ** 0.8
lc = divide_lines(ax, tree, tree["distance"] / 1000, cmap=cmocean.cm.thermal,
                  norm=Normalize(0, tree["distance"].max() / 1000), widths=w)
km_axes(ax); fig.colorbar(lc, ax=ax, shrink=0.8, label="Divide distance (km)")
ax.set_title("The divide tree by divide distance, width by topological order");

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(9, 13.2), layout="constrained")
for ax, (scheme, name, log) in zip(axs, [("strahler", "Strahler", False), ("shreve", "Shreve", True),
                                         ("topo", "Topological", False)]):
    g = Dc.to_geodataframe(seglength=60, scheme=scheme)
    g = g[g.distmin >= LIMIT]
    basemap(ax, alpha=0.4)
    v = g["order"].astype(float)
    nrm = LogNorm(1, v.max()) if log else Normalize(1, v.max())
    lc = divide_lines(ax, g, v, cmap=cmocean.cm.haline, norm=nrm, widths=0.5 + 1.8 * nrm(v) ** 1.5)
    km_axes(ax)
    cb = fig.colorbar(lc, ax=ax, shrink=0.85, label=f"{name} order")
    if not log:
        cb.locator = MaxNLocator(integer=True); cb.update_ticks()
    ax.set_title(f"{name} divide order")

## Pieces and exports

The pieces of 300 m keep the segment they belong to (`segid`), their order, mean, minimum and maximum divide distance, length, azimuth, and the flags `isbnd` and `isring`. `isring` marks pieces of closed rings. No closed-ring pieces were returned for this example. Writing a GeoPackage, a File Geodatabase or a Shapefile is one line each. A Shapefile is split into numbered parts where a 2 GB limit would be exceeded.

In [ ]:
print(f"{int(pieces.isring.sum())} pieces belong to closed rings")
pieces.head()

In [ ]:
D.to_geopackage(out / "divides_300m.gpkg", seglength=300)
D.to_shapefile(out / "divides_300m.shp", seglength=300)
print("written to", out)
sorted(p.name for p in out.iterdir() if p.stem.startswith("divides"))

## Every across-divide metric in one call

In the tutorial, the across-divide difference in hillslope relief and the divide asymmetry index were assembled from `vertdistance2stream`, `getvalue` and a few lines of arithmetic. A χ contrast would have needed the χ transform, its base level and a mapping from the channels onto the hillslopes. `across_divide_metrics` computes elevation, slope, hillslope relief, flow distance to the channel and χ, samples each at the two cells that flank every divide edge, and aggregates them per piece. It reuses the flow, stream and divide objects we already have. Without them it builds its own.

Because a χ contrast is meaningless without a common reference, the call refuses to compute χ unless the base level is stated:

In [ ]:
try:
    pydivide.across_divide_metrics(dem, flow=fd, streams=st, divides=D)
except ValueError as err:
    print(str(err)[:310], "...")

With the base level stated, one call does everything, and `describe()` returns the parametrisation in prose. Read its last sentence together with the explanation of the flag below before quoting it.

In [ ]:
tab = pydivide.across_divide_metrics(dem, flow=fd, streams=st, divides=D, baselevel=400.0,
                                     mn=0.45, a0=1.0, seglength=300, **THRESHOLD)
print(tab.describe())

Every row is a piece of divide. For each family (`elev`, `slope`, `hr` for hillslope relief, `fd` for flow distance, `chi`) there are five columns: `_mean` = (A + B) / 2, `_diff` = |A − B|, `_min`, `_max`, and the signed contrast `_sgn` = A − B. Then the divide asymmetry index `dai`, the two basins `basin_a` and `basin_b`, the reach labels of the outlets they drain through, `outlet_a` and `outlet_b`, the elevations at which those outlets leave the DEM, `outlet_z_a` and `outlet_z_b`, the flag `chi_common_base`, and a coherence per family. The parameters that made the numbers travel with the table in `tab.settings`.

Hillslope relief is the vertical height above the first downstream channel. Flow distance is the horizontal routing distance to that channel. Slope is reported in degrees. χ has units of metres here because the reference area is explicitly 1 m². For each metric, the two-sided statistics are calculated edge by edge and then averaged over each divide piece. In particular, the mean absolute edge contrast need not equal the absolute difference between the two piece means. DAI is `hr_diff / (2 * hr_mean)`, a dimensionless relief-asymmetry measure calculated from the aggregated quantities.

In [ ]:
c, gdf = tab.columns, tab.to_geodataframe()
mid = gdf.geometry.centroid
interior = (c["isbnd"] == 0) & mid.x.between(*W[:2]).values & mid.y.between(*W[2:]).values   # pieces in the window
view = interior & (c["distmin"] >= LIMIT)                 # the same view as the maps above
common = c["chi_common_base"] == 1                        # both sides reach the base level
print(list(c)[10:])
gdf.loc[view, ["segid", "basin_a", "basin_b", "chi_sgn", "hr_sgn", "dai", "outlet_a", "outlet_b",
               "outlet_z_a", "outlet_z_b", "chi_common_base"]].head()

`chi_common_base` is the check we prepared the DEM for. `baselevel=400` trims every channel at 400 m, but a river system that leaves the DEM above 400 m never reaches it, and its χ starts at its own outlet. `chi_common_base = 0` means that at least one represented outlet lies above the requested reference elevation. It does not necessarily mean that the two sides have different actual χ origins: two tributaries of the northern system share its outlet, and their χ values start from the same place, only not from 400 m. The warning that `settings` carries speaks of two different base levels, which overstates that case. Here we retain only flag 1 to apply a consistent 400 m criterion. In the window, every flagged piece has a side in the northern system, either against it or inside it. The map shows the divides between river systems.

In [ ]:
same = view & (c["outlet_a"] == c["outlet_b"])                # tributaries of one river
diff = view & (c["outlet_a"] != c["outlet_b"])                # different river systems
flagged = view & (c["chi_common_base"] == 0)
to_north = flagged & ((c["outlet_z_a"] == z_north) | (c["outlet_z_b"] == z_north))
print(tab.settings["common_base_warning"])
print(f"\nin the window: {flagged.sum():,} pieces flagged, {to_north.sum():,} of them with a side draining to the northern system, "
      f"{(flagged & same).sum():,} of them with both sides draining through the same outlet. "
      f"{diff.sum():,} pieces lie between different river systems, {(diff & common).sum():,} of them with both sides at the base level.")

fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
basemap(ax, alpha=0.45)
divide_lines(ax, gdf[view & ~diff], color="0.6", widths=0.5)
divide_lines(ax, gdf[diff & common], color="#c51b7d", widths=2.0, zorder=4)
divide_lines(ax, gdf[diff & ~common], color="#ff7f00", widths=2.4, zorder=5, linestyles=(0, (2, 1)))
km_axes(ax)
ax.legend(handles=[Line2D([], [], color="#c51b7d", lw=2, label="chi_common_base = 1: both sides reach 400 m"),
                   Line2D([], [], color="#ff7f00", lw=2.4, ls=(0, (2, 1)),
                          label=f"chi_common_base = 0: one side drains to the northern system ({z_north:.0f} m)")],
          loc="lower left", title="divides between different river systems")
ax.set_title("Which divides between river systems share the base level of 400 m");

From here on, χ contrasts are read only where `common` is true. The relief and flow-distance contrasts need no base level and are read everywhere.

### Maps of the contrasts

The first two panels are the quantities of the tutorial, the absolute relief contrast and the divide asymmetry index. The last two are signed, on a diverging scale with the white centre removed, so that a contrast near zero is still a visible line, pale blue or pale red by its sign. The line width grows with the size of the contrast. Pieces drawn in grey are excluded from the χ panel because their sides do not share the base level. They are not pieces with a small contrast.

In [ ]:
chi_ok = view & common                                   # chi only where both sides share the base level


def sym(v, mask, q=97):
    m = np.nanpercentile(np.abs(v[mask]), q)
    return Normalize(-m, m)


signed = ListedColormap(cmocean.cm.balance(np.r_[np.linspace(0.0, 0.38, 128), np.linspace(0.62, 1.0, 128)]))   # no white centre
amp = cmocean.tools.crop_by_percent(cmocean.cm.amp, 15, which="min")                                         # no white at zero
matter = cmocean.tools.crop_by_percent(cmocean.cm.matter, 15, which="min")

fig, axs = plt.subplots(2, 2, figsize=(11.5, 6.4), layout="constrained")
panels = [("hr_diff", view, "(a) Relief contrast |hr_A − hr_B|", "m", amp,
           Normalize(0, np.nanpercentile(c["hr_diff"][view], 97))),
          ("dai", view, "(b) Divide asymmetry index hr_diff / (2 hr_mean)", "", matter, Normalize(0, 1)),
          ("chi_sgn", chi_ok, "(c) Signed χ contrast χ_A − χ_B", "m", signed, sym(c["chi_sgn"], chi_ok)),
          ("hr_sgn", view, "(d) Signed relief contrast hr_A − hr_B", "m", signed, sym(c["hr_sgn"], view))]
for ax, (col, mask, title, unit, cmap, nrm) in zip(axs.flat, panels):
    basemap(ax, alpha=0.4)
    if col == "chi_sgn":
        divide_lines(ax, gdf[view & ~mask], color="0.7", widths=0.6, zorder=2.5)      # excluded, not near zero
    v = c[col][mask]
    strength = np.clip(np.abs(v) / max(abs(nrm.vmin), abs(nrm.vmax)), 0, 1)       # line width grows with the contrast
    lc = divide_lines(ax, gdf[mask], v, cmap=cmap, norm=nrm, widths=0.5 + 1.8 * strength ** 0.8)
    km_axes(ax)
    fig.colorbar(lc, ax=ax, shrink=0.85, label=unit)
    ax.set_title(title)

## Reading a signed contrast

A signed contrast is A minus B. At the first valid edge of a segment, pydivide chooses the lower-labelled reach basin as side A and then keeps A on that physical side along the segment. Reach labels can change along a segment, so A is not necessarily the lower label on every piece. The `basin_a` and `basin_b` columns identify the representative pair for a piece. The `basin_pair_mixed` and `basin_pair_share` columns show whether that pair describes all contributing edges. For pair-specific comparisons below, we use pieces with `basin_pair_mixed = 0`.

A positive `chi_sgn` means that χ is higher on side A. A negative value means that it is higher on side B. With comparable uplift, erodibility and runoff conditions and a common reference, higher χ can suggest the side expected to lose area (Willett et al., 2014; Forte and Whipple, 2018). This is an interpretation of a topographic indicator, not an observation of divide movement.

Hillslope relief supplies a complementary indicator under comparable conditions, with the sense reversed, since the lower side is the one expected to lose (Scherler and Schwanghart, 2020b). Flow distance describes another aspect of the neighbouring hillslopes, but its sign alone does not identify a universally winning or losing basin.

### Two rivers or one?

When the two sides drain through the same outlet they are tributaries of one river. Otherwise they belong to different river systems, and the table says which with `outlet_a` and `outlet_b`. The scatter compares the χ and relief indicators on the pieces that share the base level. The two agree where their signs are opposite, and `tab.disagreement()` counts it. Agreement between them is useful context, not independent validation of migration.

In [ ]:
fig = plt.figure(figsize=(11.5, 4.7), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.75, 1])
ax = basemap(fig.add_subplot(gs[0]), alpha=0.45)
divide_lines(ax, gdf[same], color="#6b8fb5", widths=0.8)
divide_lines(ax, gdf[diff], color="#c51b7d", widths=2.0, zorder=4)
km_axes(ax)
ax.legend(handles=[Line2D([], [], color="#6b8fb5", lw=1.2, label=f"tributaries of one river ({same.sum():,} pieces)"),
                   Line2D([], [], color="#c51b7d", lw=2, label=f"different river systems ({diff.sum():,} pieces)")],
          loc="lower left")
ax.set_title("Divides by the outlets of their two sides")

ax2 = fig.add_subplot(gs[1])
x, y = c["chi_sgn"], c["hr_sgn"]
lx, ly = np.nanpercentile(np.abs(x[chi_ok]), 99), np.nanpercentile(np.abs(y[chi_ok]), 99)
ax2.fill_between([0, lx], -ly, 0, color="0.93", zorder=0)
ax2.fill_between([-lx, 0], 0, ly, color="0.93", zorder=0)
ax2.scatter(x[same & chi_ok], y[same & chi_ok], s=3, color="#6b8fb5", alpha=0.45, label="one river")
ax2.scatter(x[diff & chi_ok], y[diff & chi_ok], s=8, color="#c51b7d", alpha=0.85, label="different systems")
ax2.axhline(0, color="k", lw=0.6); ax2.axvline(0, color="k", lw=0.6)
ax2.set_xlim(-lx, lx); ax2.set_ylim(-ly, ly)
ax2.set_xlabel("χ_A − χ_B (m)"); ax2.set_ylabel("hr_A − hr_B (m)")
agree = tab.disagreement()["chi_agrees_with_hr"]
ax2.set_title(f"χ and relief agree on {np.nanmean(agree[chi_ok]) * 100:.0f} % of these pieces")
ax2.legend(loc="upper right", markerscale=3)
ax2.text(0.97, 0.04, "agree", transform=ax2.transAxes, ha="right", color="0.4")
ax2.text(0.03, 0.93, "agree", transform=ax2.transAxes, ha="left", color="0.4");

## The same divides at three channel thresholds

The drainage area at which a channel begins defines the reach basins, and with them the two sides of every divide. `across_divide_multiscale` builds one divide network, here at the coarsest threshold of 10⁷ m², holds it fixed, and measures the contrasts at several thresholds, with side A on the same physical side at every scale. It reports whether the sign holds across the scales and at which scale the contrast is strongest, and the base-level flag comes per scale. `chi_sign_consistent` compares the finite, non-zero contrasts that are available, which may be fewer than three. For a clean three-threshold comparison we keep the pieces that have a usable contrast and a common base level at all three thresholds, and we say how many there are. Pieces with a zero or unavailable contrast at any threshold are left out of both maps.

In [ ]:
ms = pydivide.across_divide_multiscale(dem, thresholds=(1e5, 1e6, 1e7), baselevel=400.0,
                                       metrics=("hr", "chi"))
mc, mg = ms.columns, ms.to_geodataframe()
mmid = mg.geometry.centroid
tags = ms.settings["tags"]
spatial = ((mc["isbnd"] == 0) & mmid.x.between(*W[:2]).values & mmid.y.between(*W[2:]).values
           & (mc["distmin"] >= LIMIT))
all_common = np.logical_and.reduce([mc[f"chi_common_base{tag}"] == 1 for tag in tags])
all_scales = mc["chi_n_scales"] == len(tags)
ok = spatial & all_common & all_scales
consistent = np.asarray(mc["chi_sign_consistent"], dtype=bool)
n = int(ok.sum())
k = int((ok & consistent).sum())
print(tags)
if n:
    print(f"same sign at all three thresholds: {k:,} of {n:,} pieces ({100 * k / n:.1f} %)")
else:
    print("no pieces meet the three-threshold comparison conditions")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.9), layout="constrained")
basemap(axs[0], alpha=0.45)
divide_lines(axs[0], mg[ok & consistent], color="#0072B2", widths=1.3)
divide_lines(axs[0], mg[ok & ~consistent], color="#D55E00", widths=1.3, zorder=5)
km_axes(axs[0])
axs[0].legend(handles=[Line2D([], [], color="#0072B2", lw=2, label="same sign at all three thresholds"),
                       Line2D([], [], color="#D55E00", lw=2, label="opposite signs between thresholds")],
              loc="lower left")
axs[0].set_title(f"Is the sign of the χ contrast scale-independent? ({n:,} pieces)")
basemap(axs[1], alpha=0.45)
peak = np.asarray(mc["chi_peak_scale"], dtype=float)
names = {1e5: "10⁵ m²", 1e6: "10⁶ m²", 1e7: "10⁷ m²"}
cols = {1e5: "#E69F00", 1e6: "#56B4E9", 1e7: "#000000"}
for sc_, col in cols.items():
    divide_lines(axs[1], mg[ok & (peak == sc_)], color=col, widths=1.3)
km_axes(axs[1])
axs[1].legend(handles=[Line2D([], [], color=col, lw=2, label=names[s_]) for s_, col in cols.items()],
              loc="lower left", title="strongest χ contrast at")
axs[1].set_title("The threshold at which the χ contrast peaks");

## Competing basin pairs and their key

A signed contrast compares two named basins, so the pair of basins is a natural unit of analysis. A pair exists relative to a partition of the stream network, and the partition follows the channel threshold. `reach_basin_labels` gives the reach basins of any partition. The partitions are nested: a coarser threshold merges basins and never splits them, so the divides of a coarser partition are a subset of the network we built. The pair key names the partition by a power of ten in square metres. Here we look at the partitions at 10⁶, 10⁷ and 10⁸ m² in the window. The channels on the maps are those of the 1,000-cell network, held fixed for context, so they do not change with the displayed partition.

The example pairs are chosen by rule rather than by hand. For the profiles below we take the first pair of the 10⁷ m² partition, in order of shared divide length, whose shared divide lies almost entirely inside the window. For the 10⁸ m² partition we take the longest pair inside the window whose two basins the DEM holds whole, since a basin that touches the edge of the DEM or nodata is truncated there. The labels named below belong to this partition of this DEM, not to anything else.

In [ ]:
L = {e: pydivide.reach_basin_labels(dem, 10.0 ** e, fd=fd) for e in (6, 7, 8)}
pairs = {e: pydivide.basin_pairs(D, dem, labels=L[e], threshold=10.0 ** e) for e in L}

ca, cb = edge_cells(D.grid, D.edge_nodes)
x1, y1 = D.grid.node_xy(D.edge_nodes[:, 0]); x2, y2 = D.grid.node_xy(D.edge_nodes[:, 1])
edge_xy = np.stack([np.stack([x1, y1], 1), np.stack([x2, y2], 1)], 1)
edge_in_window = (x1 >= W[0]) & (x1 <= W[1]) & (y1 >= W[2]) & (y1 <= W[3])


def edge_labels(lab):
    flat = lab.ravel()
    return (np.where(ca >= 0, flat[np.maximum(ca, 0)], -1), np.where(cb >= 0, flat[np.maximum(cb, 0)], -1))


def truncated(lab):
    # labels of the basins that touch the edge of the DEM or a nodata cell
    nan = np.isnan(np.asarray(dem.z, float))
    edge = nan.copy(); edge[[0, -1], :] = True; edge[:, [0, -1]] = True
    for shift, axis in ((1, 0), (-1, 0), (1, 1), (-1, 1)):
        edge |= np.roll(nan, shift, axis=axis)
    return set(np.unique(lab[edge]).tolist()) - {0}


def longest_in_window(e, exclude=()):
    # the pair of partition e with the longest shared divide inside the window, skipping excluded basins
    la, lb = edge_labels(L[e])
    best, best_len = None, 0.0
    for lo, hi in zip(pairs[e].basin_lo, pairs[e].basin_hi):
        if lo in exclude or hi in exclude:
            continue
        mine = (np.minimum(la, lb) == lo) & (np.maximum(la, lb) == hi) & edge_in_window
        if mine.sum() * dem.cellsize > best_len:
            best, best_len = (int(lo), int(hi)), mine.sum() * dem.cellsize
    return best, best_len


def first_inside_window(e, share=0.95):
    # the first pair of partition e, longest shared divide first, with at least this share of it in the window
    la, lb = edge_labels(L[e])
    for lo, hi in zip(pairs[e].basin_lo, pairs[e].basin_hi):
        mine = (np.minimum(la, lb) == lo) & (np.maximum(la, lb) == hi)
        if edge_in_window[mine].mean() >= share:
            return (int(lo), int(hi)), (mine & edge_in_window).sum() * dem.cellsize


pair7, len7 = first_inside_window(7)
trunc8 = truncated(L[8])
pair8, len8 = longest_in_window(8, exclude=trunc8)
for e in L:
    print(f"partition 10^{e} m²: {len(np.unique(L[e][L[e] > 0])):,} reach basins, {len(pairs[e]):,} competing pairs in the DEM")
print(f"profile example: pair {pair7} of the 10^7 m² partition, {len7 / 1000:.1f} km of shared divide in the window")
print(f"10^8 m² example: pair {pair8}, {len8 / 1000:.1f} km of shared divide in the window; "
      f"{len(trunc8)} of the 10^8 m² basins touch the edge of the DEM and are not eligible")

In [ ]:
cmaps = {e: ListedColormap(np.random.default_rng(e).permutation(plt.cm.tab20(np.linspace(0, 1, 20)))) for e in L}
sup = {6: "⁶", 7: "⁷", 8: "⁸"}


def partition_panel(ax, e, extent=W, highlight=None):
    la, lb = edge_labels(L[e])
    between = (la > 0) & (lb > 0) & (la != lb)
    lab = L[e].astype(float); lab[lab < 1] = np.nan
    if highlight:
        lab[(L[e] == highlight[0]) | (L[e] == highlight[1])] = np.nan   # the pair is drawn in its own colours
    basemap(ax, alpha=0.6, extent=extent)
    ax.imshow(lab % 20, extent=EXT, cmap=cmaps[e], alpha=0.5, interpolation="nearest", zorder=1)
    ax.add_collection(LineCollection(edge_xy[between], colors="k", linewidths=0.6, zorder=3))
    if highlight:
        for lbl, colr in zip(highlight, ("#1f77b4", "#d62728")):
            ax.imshow(np.where(L[e] == lbl, 1.0, np.nan), extent=EXT, cmap=ListedColormap([colr]),
                      alpha=0.65, interpolation="nearest", zorder=2)
        mine = (np.minimum(la, lb) == highlight[0]) & (np.maximum(la, lb) == highlight[1])
        ax.add_collection(LineCollection(edge_xy[mine], colors="#ffd92f", linewidths=4.5, zorder=5))
        ax.add_collection(LineCollection(edge_xy[mine], colors="k", linewidths=1.2, zorder=6))
    km_axes(ax, extent)
    n_in = len(np.unique(L[e][in_window][L[e][in_window] > 0]))
    return n_in


fig, axs = plt.subplots(3, 1, figsize=(9.5, 15.6), layout="constrained")
for ax, e in zip(axs, (6, 7, 8)):
    n_in = partition_panel(ax, e, highlight=pair7 if e == 7 else (pair8 if e == 8 else None))
    extra = {7: f"; the profile pair {pair7}, A blue and B red", 8: f"; pair {pair8}, A blue and B red"}.get(e, "")
    ax.set_title(f"Partition 10{sup[e]} m²: {n_in:,} reach basins in the window, {len(pairs[e]):,} competing pairs in the DEM{extra}")

`basin_pairs` lists every competing pair of a partition, longest shared divide first. Each pair has a key, E·10¹⁴ + lo·10⁷ + hi, where 10^E m² is the threshold and lo and hi are the two labels. The key reads from left to right and names the pair within that partition of this network. `decode_pair_key` gives the threshold and the labels back.

In [ ]:
for e, pr in ((7, pair7), (8, pair8)):
    row = pairs[e][(pairs[e].basin_lo == pr[0]) & (pairs[e].basin_hi == pr[1])]
    key = int(row.pair_key.iloc[0])
    print(key, "=", pydivide.readable_pair_key(key), "->", pydivide.decode_pair_key(key))
pairs[7][(pairs[7].basin_lo == pair7[0]) & (pairs[7].basin_hi == pair7[1])]

For tables of contrasts, `pair_table` gathers the pieces of every pair, re-expresses each signed value as side A minus side B of that pair, and averages them, so that two pieces describing the same contrast with opposite orientations add up instead of cancelling. The contrasts of the 10⁶ m² pairs come from the same divides measured with that threshold, which is one more call, and we keep the unmixed pieces so that each piece belongs to one pair. `basin_polygons`, `basin_points` and `pair_side_points` turn the pairs into GIS layers, so that selecting one pair in QGIS highlights the two competing basins and the divide between them together.

In [ ]:
tab6 = pydivide.across_divide_metrics(dem, flow=fd, divides=D, baselevel=400.0, threshold=1e6,
                                      metrics=("hr", "chi"), seglength=300)
c6 = tab6.columns
ok6 = view & (c6["chi_common_base"] == 1) & (c6["basin_pair_mixed"] == 0)
ptab = pydivide.pair_table(c6["basin_a"][ok6], c6["basin_b"][ok6],
                           values={"chi_sgn": c6["chi_sgn"][ok6], "hr_sgn": c6["hr_sgn"][ok6]},
                           scale="1e6")
keep = np.union1d(ptab["side_a"], ptab["side_b"])
polys = pydivide.basin_polygons(L[6], D.grid, crs=D.crs, keep=keep)
print(f"{len(ptab):,} pairs at 10⁶ m² in the window, {len(polys):,} basin polygons for their {len(keep):,} basins")
ptab.sort_values("n_pieces", ascending=False)[["pair_key", "side_a", "side_b", "n_pieces", "n_flipped",
                                                "chi_sgn_mean", "hr_sgn_mean"]].head()

## The divide of a pair, along and across

`basin_pair_profiles` assembles the edges shared by a selected pair into one or more paths. It returns an along-divide profile and normal profiles at the chosen station spacing. In this profile object, `label_a` is the lower basin label and `label_b` is the higher one. Negative offsets point locally towards `label_a` and positive offsets towards `label_b`. This convention should be aligned explicitly with any signed table values being compared.

Here we inspect the highlighted 10⁷ m² pair with stations every kilometre and sections 1.5 km long. A straight section can cross another boundary farther from its station, so its entire length need not remain within the original two basins. The profiles let us examine the terrain behind the basin contrast.

In [ ]:
p = pydivide.basin_pair_profiles(D, dem, pair=pair7, labels=L[7], threshold=1e7,
                                 spacing=1000, half_length=750)
path = p.paths[0]
print(p.readable, f"| label_a = {p.label_a} (negative offsets), label_b = {p.label_b} (positive offsets) | "
      f"{p.length / 1000:.1f} km of shared divide in {len(p.paths)} path(s) | {p.n_stations} stations")

nrm = path.normals
fig, ax = plt.subplots(figsize=(8.5, 7.5), layout="constrained")
plot_basin_pair(L[7], pair7, dem=dem, D=D, ax=ax, pad_cells=25)
xl, yl = ax.get_xlim(), ax.get_ylim()                  # the channels would expand the axes to the whole DEM
st.plot(ax=ax, color="#4d88ff", linewidth=0.6, alpha=0.9, zorder=2)
ax.set_xlim(xl); ax.set_ylim(yl)
for (sx, sy), (vx, vy), name in zip(nrm["station_xy"], nrm["normal"], nrm["station_label"]):
    ax.plot([sx - 750 * vx, sx + 750 * vx], [sy - 750 * vy, sy + 750 * vy], color="#ffd92f", lw=2.2, zorder=6,
            solid_capstyle="round")
    ax.text(sx + 860 * vx, sy + 860 * vy, name, color="k", fontweight="bold", ha="center", va="center", zorder=7,
            bbox=dict(boxstyle="circle,pad=0.15", fc="w", ec="none", alpha=0.8))
ax.set_title(f"Pair {p.readable}: the shared divide and the sections across it");

In [ ]:
fig = p.plot_along(stations=True, title=f"Along-divide profile of pair {p.readable}, with the stations")

In [ ]:
fig = p.plot_normals(layout="stacked")

A swath of a chosen width replaces each section with statistics across the width. Here every third section (b, e, h and k), 600 m wide. `p.save(...)` writes the figures, the tables and a GeoPackage of the pair.

In [ ]:
sw = pydivide.basin_pair_profiles(D, dem, pair=pair7, labels=L[7], threshold=1e7,
                                  spacing=3000, half_length=750, swath_width=600)
fig = sw.plot_normals(layout="grid", ncols=4)
fig.suptitle(f"Swaths 600 m wide across the divide of pair {sw.readable}: median, quartiles and range", fontsize=10)
fig.set_size_inches(11.5, 3.6)
for ax, s in zip(fig.axes, sw.paths[0].normals["station_s"]):      # name each swath after its section on the map
    k = int(np.argmin(np.abs(nrm["station_s"] - s)))
    ax.set_title(f"section {nrm['station_label'][k]}, {s / 1000:g} km", fontsize=8, fontweight="bold")

## The base level moves the sign

Raising the base level removes the lowest reaches of every channel. Where both sides of a divide lose the same reaches their χ contrast is unchanged. Where the removed reaches differ, the contrast can change sign. Because the base level is an argument of the call, testing this is a loop. We raise it from 400 m in steps and count, among the pieces comparable at both levels and whose sides share both levels, how often the sign of χ_A − χ_B reverses. The counts behind each percentage are printed with it.

In [ ]:
ladder = [400, 500, 600, 700]
runs = {b: pydivide.across_divide_metrics(dem, flow=fd, streams=st, divides=D, baselevel=float(b),
                                          metrics=("chi",), seglength=300, **THRESHOLD).columns for b in ladder}
ref = runs[400]["chi_sgn"]
rows = []
for b in ladder[1:]:
    both = (chi_ok & (runs[b]["chi_common_base"] == 1) & np.isfinite(ref) & np.isfinite(runs[b]["chi_sgn"])
            & (ref != 0) & (runs[b]["chi_sgn"] != 0))
    flip = both & (np.sign(ref) != np.sign(runs[b]["chi_sgn"]))
    rows.append((b, flip[same].sum(), both[same].sum(), flip[diff].sum(), both[diff].sum(), flip))
for b, fs, ns, fdf, nd, _ in rows:
    print(f"{b} m: reversed on {fs:,} of {ns:,} one-river pieces ({100 * fs / ns:.2f} %) "
          f"and on {fdf:,} of {nd:,} different-system pieces ({100 * fdf / nd:.2f} %)")

In [ ]:
flip = rows[-1][5]
fig = plt.figure(figsize=(11.5, 4.4), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.8])
ax = fig.add_subplot(gs[0])
ax.plot([r[0] for r in rows], [100 * r[3] / r[4] for r in rows], "o-", color="#c51b7d", lw=2, label="different river systems")
ax.plot([r[0] for r in rows], [100 * r[1] / r[2] for r in rows], "o-", color="#6b8fb5", lw=2, label="tributaries of one river")
ax.set_xlabel("Base level (m)"); ax.set_ylabel("Pieces whose χ contrast reverses (%)")
ax.set_title("Relative to a base level of 400 m"); ax.legend(); ax.grid(alpha=0.3)

ax2 = basemap(fig.add_subplot(gs[1]), alpha=0.45)
divide_lines(ax2, gdf[chi_ok], color="0.6", widths=0.5)
divide_lines(ax2, gdf[flip & diff], color="#c51b7d", widths=2.6, zorder=5)
divide_lines(ax2, gdf[flip & same], color="#6b8fb5", widths=2.2, zorder=4)
km_axes(ax2)
ax2.legend(handles=[Line2D([], [], color="#c51b7d", lw=2.5, label="different river systems"),
                    Line2D([], [], color="#6b8fb5", lw=2.5, label="tributaries of one river")],
           loc="lower left", title="sign reversed")
ax2.set_title(f"Pieces whose χ contrast reverses between 400 and {ladder[-1]} m");

## The other `DIVIDEobj` operations

`pydivide.divideops` reproduces the remaining methods of `DIVIDEobj` on the same definitions: besides `cleanedges`, also `removeshortdivides`, `shrink`, `dist2node`, `jctangle` and `jctcon`. `getvalue` and `asymmetry` live in `pydivide.divmetrics`.

In [ ]:
variants = [("removeshortdivides, 1 km", ops.remove_short_divides(D, FX, 1000.0)),
            ("shrink by distance, 2 km", ops.shrink(D, FX, "distance", 2000.0))]
angles = ops.junction_angles(D)
jx, jy = D.grid.node_xy(np.asarray(D.junctions))
cx, cy = np.mean(W[:2]), np.mean(W[2:])
start = np.asarray(D.junctions)[np.argmin((jx - cx) ** 2 + (jy - cy) ** 2)]
sx0, sy0 = D.grid.node_xy(np.array([start]))
d_start = ops.dist_to_node(D, np.array([start]), penalty=15000.0)

fig, axs = plt.subplots(2, 2, figsize=(11.5, 6.4), layout="constrained")
for ax, (name, Dv) in zip(axs.flat[:2], variants):
    basemap(ax, alpha=0.4)
    gv = Dv.to_geodataframe(seglength=60)
    divide_lines(ax, gv[gv.distmin >= LIMIT], color="#252525", widths=0.7)
    km_axes(ax)
    ax.set_title(f"{name}: {Dv.n_segments:,} segments")
ax = axs.flat[2]
basemap(ax, alpha=0.4)
nx_, ny_ = D.grid.node_xy(np.asarray(D.network.seg_nodes))
okd = np.isfinite(d_start)
sc = ax.scatter(nx_[okd], ny_[okd], c=d_start[okd] / 1000, s=0.8, cmap=cmocean.cm.thermal, zorder=3)
ax.scatter(sx0, sy0, marker="*", s=220, color="w", edgecolor="k", zorder=5)
km_axes(ax)
fig.colorbar(sc, ax=ax, shrink=0.85, label="km")
ax.set_title("dist2node: divide distance re-rooted at a junction")
ax = axs.flat[3]
vals = np.concatenate([angles[k] for k in ("a", "b", "c")])     # the three angles at every degree-3 junction
ax.hist(vals[np.isfinite(vals)], bins=np.arange(0, 241, 10), color="#4c72b0")
ax.axvline(120, color="k", lw=0.8, ls="--")
ax.set_xlabel("Angle between divide segments at degree-3 junctions (degrees)"); ax.set_ylabel("Count")
ax.set_title(f"jctangle: {len(angles['a']):,} junctions");

## Exports that carry their settings

`tab.to_file` writes the table as GeoPackage, File Geodatabase or Shapefile, with a settings document beside it. A GeoPackage also stores that document inside the file, so it travels on its own. The other formats rely on the sidecar, which must travel with them. For GeoPackage, `read_export_settings` reads the settings back from the file alone, so the base level, threshold and concavity behind a table stay with it.

In [ ]:
written = tab.to_file(out / "metrics_300m.gpkg")
print(*[p.name for p in written], sep="\n")
settings = pydivide.read_export_settings(out / "metrics_300m.gpkg")
{k: settings["settings"][k] for k in ("threshold", "threshold_units", "baselevel_m", "mn", "a0", "seglength_m", "side_rule")}

## Summary

The example leaves three practical lessons:

- A useful map window can be smaller than the domain needed for hydrologic analysis.
- Signed contrasts identify the compared sides, while their interpretation depends on the chosen parameters and comparison conditions.
- Basin pairs and along/across profiles connect numerical contrasts to terrain we can inspect.

Try changing the χ base level and compare the eligible population before interpreting sign reversals. Then choose another basin pair and inspect its profiles. Which features help explain its metric contrast, and which questions remain open?

The table below provides a reference for applying these steps to another landscape.

| capability | in this notebook |
|---|---|
| divide network, distance and the three orders | `divide_network` |
| edge effects | the `isbnd` flag, or `divideops.clean_edges` |
| the divide tree beyond a distance | a filter on `distmin` |
| pieces and exports | `to_geodataframe`, `to_geopackage`, `to_shapefile`, `to_filegdb` |
| every across-divide metric, with its parameters | `across_divide_metrics(..., baselevel=...)`, `describe()` |
| do both sides reach the base level? | `chi_common_base`, `outlet_z_a`, `outlet_z_b` |
| signed contrasts between named basins | the `_sgn` columns, `basin_a`, `basin_b`, `basin_pair_mixed` |
| tributaries of one river, or different rivers? | `outlet_a`, `outlet_b` |
| several channel thresholds at once | `across_divide_multiscale` |
| competing basin pairs and their key | `reach_basin_labels`, `basin_pairs`, `pair_table`, `basin_polygons` |
| the divide of a pair, along and across | `basin_pair_profiles`, `plot_basin_pair` |
| the other `DIVIDEobj` methods | `pydivide.divideops`, `pydivide.divmetrics` |
| settings that travel with the file | `to_file`, `read_export_settings` |

We are exploring how these tools can support investigations of drainage reorganisation and possible river capture.

## References and additional information

European Space Agency: Copernicus Global Digital Elevation Model (GLO-30), https://registry.opendata.aws/copernicus-dem/, 2021.

Forte, A. M. and Whipple, K. X.: Criteria and tools for determining drainage divide stability, Earth Planet. Sc. Lett., 493, 102–117, https://doi.org/10.1016/j.epsl.2018.04.026, 2018.

Peifer, D.: pydivide: drainage divide networks in Python beyond the 32-bit limit, Zenodo [code], https://doi.org/10.5281/zenodo.23475347, 2026.

Perron, J. T. and Royden, L.: An integral approach to bedrock river profile analysis, Earth Surf. Proc. Land., 38, 570–576, https://doi.org/10.1002/esp.3302, 2013.

Scherler, D.: Introduction to @DIVIDEobj, TopoToolbox blog, https://topotoolbox.wordpress.com/2020/05/29/introduction-to-divideobj/, 2020.

Scherler, D. and Schwanghart, W.: Drainage divide networks – Part 1: Identification and ordering in digital elevation models, Earth Surf. Dynam., 8, 245–259, https://doi.org/10.5194/esurf-8-245-2020, 2020a.

Scherler, D. and Schwanghart, W.: Drainage divide networks – Part 2: Response to perturbations, Earth Surf. Dynam., 8, 261–274, https://doi.org/10.5194/esurf-8-261-2020, 2020b.

Willett, S. D., McCoy, S. W., Perron, J. T., Goren, L., and Chen, C.-Y.: Dynamic reorganization of river basins, Science, 343, 1248765, https://doi.org/10.1126/science.1248765, 2014.